This code loads in multiple headfixed sessions from the same set of mice and generates a dataframe that is then used for mixed effects model analysis. 

All sessions for a given mouse are concatenated and a combined baseline is generated by extracting the 5s prior to spout extension for each trial. Z scores are calculated using the mean and sd of this combined baseline.

Used to analyze lick data and photometry signals from headfixed OHRBETS system (https://doi.org/10.7554/eLife.86183).
Curve fit and motion correction are based on Simpson et al. 2024 (https://doi.org/10.1016/j.neuron.2023.11.016). 
Cursor AI software was used to assist with writing parts of this code.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
# --- Cohort-level settings ---
output_name = "Cohort1"
data_output = "./Output_dataframes"
mean_trace_output = "./mean_traces"

#program parameters (shared)
trials = 60
trial_length = 3000  # total time spout is extended, in ms
buffer_duration = 0  # duration of no-flow time at start and end of trial in ms
lick_codes = [31, 32, 33, 34, 35, 431, 432, 433, 434, 435]  # defining which codes are for licks
n_spouts = 5  # number of spouts used

#group parameters
mouse_group_mapping = {
    "0001": "GCaMP",
    "0002": "GCaMP",
    "0003": "GCaMP",
    "0004": "GCaMP"
}

# --- One dict per session. 
# Edit paths, spout_mapping, and photometry stream / TTL names per day.
SESSIONS = [
    {
        "label": "WRD1",
        "lick_filepath": r"Cohort1\water restricted\D1\Arduino output\CSVs",
        "data_directory": r"Cohort1\water restricted\D1\Photometry data",
        "acq_day": 1,
        "restriction_state": 1,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            4: "10% Sucrose",
            1: "Saccharine",
            2: "20% Sucrose",
            3: "Water",
            0: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_464D", "ttl_epoc": "PtC3"},
        },
    },
    {
        "label": "WRD2",
        "lick_filepath": r"Cohort1\water restricted\D2\Arduino output\CSVs",
        "data_directory": r"Cohort1\water restricted\D2\Photometry data",
        "acq_day": 2,
        "restriction_state": 1,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            0: "10% Sucrose",
            2: "Saccharine",
            3: "20% Sucrose",
            4: "Water",
            1: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_464D", "ttl_epoc": "PtC3"},
        },
    },
    {
        "label": "WRD3",
        "lick_filepath": r"Cohort1\water restricted\D3\Arduino output\CSVs",
        "data_directory": r"Cohort1\water restricted\D3\Photometry data",
        "acq_day": 3,
        "restriction_state": 1,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            1: "10% Sucrose",
            3: "Saccharine",
            0: "20% Sucrose",
            2: "Water",
            4: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_464D", "ttl_epoc": "PtC3"},
        },
    },
    {
        "label": "WRD4",
        "lick_filepath": r"Cohort1\water restricted\D4\Arduino output\CSVs",
        "data_directory": r"Cohort1\water restricted\D4\Photometry data",
        "acq_day": 4,
        "restriction_state": 1,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            2: "10% Sucrose",
            4: "Saccharine",
            1: "20% Sucrose",
            3: "Water",
            0: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_464D", "ttl_epoc": "PtC3"},
        },
    },
    {
        "label": "WRD5",
        "lick_filepath": r"Cohort1\water restricted\D5\Arduino output\CSVs",
        "data_directory": r"Cohort1\\water restricted\D5\Photometry data",
        "acq_day": 5,
        "restriction_state": 1,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            4: "10% Sucrose",
            1: "Saccharine",
            2: "20% Sucrose",
            0: "Water",
            3: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_464D", "ttl_epoc": "PtC3"},
        },
    },
    {
        "label": "FRD1",
        "lick_filepath": r"Cohort1\food restricted\D1\Arduino output\CSVs",
        "data_directory": r"Cohort1\food restricted\D1\Photometry data",
        "acq_day": 6,
        "restriction_state": 2,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            3: "10% Sucrose",
            0: "Saccharine",
            2: "20% Sucrose",
            4: "Water",
            1: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_463C", "ttl_epoc": "PtC2"},
        },
    },
    {
        "label": "FRD2",
        "lick_filepath": r"Cohort1\food restricted\D2\Arduino output\CSVs",
        "data_directory": r"Cohort1\food restricted\D2\Photometry data",
        "acq_day": 7,
        "restriction_state": 2,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            4: "10% Sucrose",
            1: "Saccharine",
            3: "20% Sucrose",
            2: "Water",
            0: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_463C", "ttl_epoc": "PtC2"},
        },
    },
    {
        "label": "FRD3",
        "lick_filepath": r"Cohort1\food restricted\D3\Arduino output\CSVs",
        "data_directory": r"Cohort1\food restricted\D3\Photometry data",
        "acq_day": 8,
        "restriction_state": 2,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            0: "10% Sucrose",
            2: "Saccharine",
            1: "20% Sucrose",
            3: "Water",
            4: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_463C", "ttl_epoc": "PtC2"},
        },
    },
    {
        "label": "FRD4",
        "lick_filepath": r"Cohort1\food restricted\D4\Arduino output\CSVs",
        "data_directory": r"Cohort1\food restricted\D4\Photometry data",
        "acq_day": 9,
        "restriction_state": 2,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            1: "10% Sucrose",
            4: "Saccharine",
            2: "20% Sucrose",
            0: "Water",
            3: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_463C", "ttl_epoc": "PtC2"},
        },
    },
    {
        "label": "FRD5",
        "lick_filepath": r"Cohort1\food restricted\D5\Arduino output\CSVs",
        "data_directory": r"Cohort1\food restricted\D5\Photometry data",
        "acq_day": 10,
        "restriction_state": 2,  # 1=water restricted, 2=food restricted
        "spout_mapping": {
            2: "10% Sucrose",
            3: "Saccharine",
            0: "20% Sucrose",
            1: "Water",
            4: "Null",
        },
        # TDT: attribute names on block.streams / block.epocs (e.g. _462B, PtC1)
        "photometry": {
            "mouse1": {"green_stream": "_461A", "ttl_epoc": "PtC0"},
            "mouse2": {"green_stream": "_463C", "ttl_epoc": "PtC2"},
        },
    },
]

def _tdt_stream(block, name):
    return getattr(block.streams, name)

def _tdt_epoc_onset(block, name):
    return getattr(block.epocs, name).onset

In [ ]:
# Define a function to perform the adjustments for each trial timestamp
def adjust_licktimes(trial_timestamp):
    # Identify all timestamps in licktimes within the time window of the trial timestamp
    relevant_licktimes = licktimes[(licktimes['timestamp'] >= trial_timestamp) & 
                                   (licktimes['timestamp'] <= trial_timestamp + time_window)]
    
    # Adjust the relevant licktimes relative to the trial timestamp
    adjusted_licktimes = relevant_licktimes['timestamp'] - trial_timestamp
    
    # Convert the adjusted licktimes to a list
    adjusted_licktimes_list = adjusted_licktimes.tolist()
    
    # Return the list of adjusted licktimes
    return adjusted_licktimes_list

In [ ]:
list_of_dfs = []

for sess in SESSIONS:
    lick_filepath = sess["lick_filepath"]
    spout_mapping = sess["spout_mapping"]
    acq_day = sess["acq_day"]
    restriction_state = sess["restriction_state"]

    file_list = os.listdir(lick_filepath)
    data_list = {}
    session_mouseIDs = []

    for file in file_list:
        if file.endswith(".csv"):
            file_name = os.path.basename(file)
            mouse_data = pd.read_csv(os.path.join(lick_filepath, file), sep=" ", header=None)
            mouse_data.columns = ["event_tag", "timestamp"]
            mouse_id = file_name.split("_")[3].split(".")[0]
            mouse_data["mouse_id"] = mouse_id
            data_name = "data_" + mouse_id
            globals()[data_name] = mouse_data
            data_list[data_name] = mouse_data
            session_mouseIDs.append(mouse_id)

    for mouse in session_mouseIDs:
        data_name = "data_" + mouse
        data = data_list[data_name]

        spout_extensions = data[data["event_tag"] == 13]

        if len(spout_extensions) > trials:
            spout_extensions = spout_extensions.iloc[1:]
            spout_extensions.reset_index(drop=True, inplace=True)

        spout_position = data[data["event_tag"] == 127]

        if len(spout_position) > (2 * trials):
            spout_position = spout_position.iloc[1:]

        trial_list = spout_position[spout_position["timestamp"] <= 5]
        trial_list = trial_list.rename(columns={"timestamp": "spout_position"})
        sp_codes = pd.to_numeric(trial_list["spout_position"], errors="coerce").round().astype(int)
        trial_list["spout_position"] = sp_codes.map(spout_mapping)
        trial_list.reset_index(drop=True, inplace=True)
        columnlist = ["mouse_id", "spout_position"]
        trial_list = trial_list[columnlist]

        trial_list["timestamp"] = spout_extensions["timestamp"]

        licktimes = data[data["event_tag"].isin(lick_codes)]

        time_window = trial_length + 50

        trial_list["adjusted_licktimes"] = trial_list["timestamp"].apply(adjust_licktimes)

        trial_list["day"] = acq_day
        trial_list["restriction_state"] = restriction_state

        list_of_dfs.append(trial_list)

all_trial_list = pd.concat(list_of_dfs, ignore_index=True)

# Create a new column 'group' based on the mouse_id
all_trial_list['group'] = all_trial_list['mouse_id'].map(mouse_group_mapping)

In [ ]:
# Define a function to plot lick rate over time during each trial
def create_histogram(timestamps):
    # Create histogram bins
    bins = np.arange(0, trial_length+101, 100)
    # Create histogram
    hist, _ = np.histogram(timestamps, bins=bins)
    return hist

# Apply the create_histogram function to each list of timestamps
all_trial_list['histogram'] = all_trial_list['adjusted_licktimes'].apply(create_histogram)

#Convert to Hz by multiplying the histo by 10 (currently licks per 100 ms bin)
all_trial_list['histogram_Hz'] = all_trial_list['histogram'] * 10

In [ ]:
# Making plots of lick rate.
# Within a panel, mean histogram_Hz per spout_position, all spouts on the same axes, color = spout.

from matplotlib.lines import Line2D

RESTRICTION_LABEL = {1: "water restricted", 2: "food restricted"}

avg_by_spout = (
    all_trial_list.groupby(
        ["mouse_id", "day", "restriction_state", "spout_position"], dropna=False
    )["histogram_Hz"]
    .mean()
    .reset_index()
)

x_bins = np.arange(0, trial_length + 100, 100)
n_cols = 5

mice = sorted(avg_by_spout["mouse_id"].dropna().unique(), key=str)
rests = sorted(avg_by_spout["restriction_state"].dropna().unique())

for mid in mice:
    for rest in rests:
        sub = avg_by_spout[
            (avg_by_spout["mouse_id"] == mid)
            & (avg_by_spout["restriction_state"] == rest)
        ]
        if sub.empty:
            continue

        days = sorted(sub["day"].dropna().unique())
        spouts = sorted(sub["spout_position"].dropna().unique(), key=str)
        if not days or not spouts:
            continue

        n_days = len(days)
        n_rows = int(math.ceil(n_days / n_cols))

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(3.2 * n_cols, 2.7 * n_rows),
            sharex=True,
            sharey=True,
            squeeze=False,
        )

        sp_colors = {sp: plt.cm.tab10(i % 10) for i, sp in enumerate(spouts)}

        for i, day in enumerate(days):
            r, c = divmod(i, n_cols)
            ax = axes[r][c]
            day_sub = sub[sub["day"] == day]
            for sp in spouts:
                d = day_sub[day_sub["spout_position"] == sp]
                if d.empty:
                    continue
                y = d["histogram_Hz"].iloc[0]
                ax.plot(x_bins, y, color=sp_colors[sp], lw=1.5)
            ax.set_title(f"Day {day}", fontsize=9)

        for j in range(n_days, n_rows * n_cols):
            r, c = divmod(j, n_cols)
            axes[r][c].set_visible(False)

        try:
            rk = int(rest)
        except (TypeError, ValueError):
            rk = rest
        rlab = RESTRICTION_LABEL.get(rk, str(rest))
        fig.suptitle(f"Mouse {mid} | {rlab}", fontsize=12, y=1.02)

        handles = [
            Line2D([0], [0], color=sp_colors[s], lw=1.5, label=str(s)) for s in spouts
        ]
        fig.legend(
            handles=handles,
            loc="center left",
            bbox_to_anchor=(1.0, 0.5),
            borderaxespad=0.35,
            frameon=False,
            title="Spout",
        )

        for c in range(n_cols):
            ax = axes[n_rows - 1][c]
            if ax.get_visible():
                ax.set_xlabel("Time (ms)")
        for r in range(n_rows):
            ax = axes[r][0]
            if ax.get_visible():
                ax.set_ylabel("Lick rate (Hz)")

        plt.tight_layout(rect=[0, 0, 0.95, 0.97])
        plt.show()

### average lick rate across groups

In [ ]:
# Per mouse: mean lick rate for each day × restriction × spout × group.
# Then average across mice, keeping day and restriction_state separate.
mouse_avg_hist = (
    all_trial_list.groupby(
        ["mouse_id", "day", "restriction_state", "spout_position", "group"],
        dropna=False,
    )["histogram_Hz"]
    .mean()
    .reset_index()
)

group_avg_hist = (
    mouse_avg_hist.groupby(
        ["group", "spout_position", "day", "restriction_state"]
    )["histogram_Hz"]
    .apply(lambda x: np.mean(np.vstack(x), axis=0))
    .reset_index()
)

RESTRICTION_LABEL = {1: "water restricted", 2: "food restricted"}
x_bins = np.arange(0, trial_length + 100, 100)

# One figure per (day, restriction_state, group); lines = spout (cohort mean across mice)
for (day, rest), dr_data in group_avg_hist.groupby(
    ["day", "restriction_state"], sort=True
):
    try:
        rlab = RESTRICTION_LABEL.get(int(rest), str(rest))
    except (TypeError, ValueError):
        rlab = str(rest)
    for group_name, group_data in dr_data.groupby("group"):
        fig, ax = plt.subplots(figsize=(4, 3))
        for spout_position, hist in group_data.groupby("spout_position"):
            ax.plot(x_bins, hist.iloc[0]["histogram_Hz"], label=spout_position)
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Lick Rate (Hz)")
        ax.set_title(f"{group_name} | Day {day} | {rlab}\n(mean across mice)")
        ax.legend()
        plt.show()

In [ ]:
#adding total lick calculations to our dataframe
# Define a function to calculate total number of licks
def total_licks(licktimes):
    return len(licktimes)

# Define a function to calculate number of licks during flow time if paradigm has buffers
def licks_during_flow(licktimes):
    return sum(buffer_duration <= ts <= buffer_duration+trial_length for ts in licktimes)

# Apply the functions to each row and assign the results to new columns
all_trial_list['total_licks'] = all_trial_list['adjusted_licktimes'].apply(total_licks)
#all_trial_list['licks_during_flow'] = all_trial_list['adjusted_licktimes'].apply(licks_during_flow)

## Photometry analysis

In [ ]:
import tdt
from scipy.optimize import curve_fit

### Get list of folders, extract their data

In [ ]:
# List tank folders per session and read TDT blocks. Order matches SESSIONS, then folder order.
masterdat = []
for sess in SESSIONS:
    data_directory = sess["data_directory"]
    folderlist = [
        item
        for item in os.listdir(data_directory)
        if os.path.isdir(os.path.join(data_directory, item))
    ]
    for foldername in folderlist:
        dat = {}

        if len(foldername) == 18:
            dat["mouse1"] = foldername[0:4]
            dat["mouse2"] = "0000"
            dat["date"] = foldername[5:11]
        else:
            dat["mouse1"] = foldername[0:4]
            dat["mouse2"] = foldername[5:9]
            dat["date"] = foldername[10:16]

        dat["blockpath"] = foldername
        dat["data"] = tdt.read_block(os.path.join(data_directory, dat["blockpath"]))
        dat["acq_day"] = sess["acq_day"]
        dat["restriction_state"] = sess["restriction_state"]
        dat["session_label"] = sess.get("label", str(sess["acq_day"]))
        dat["photometry"] = sess["photometry"]

        masterdat.append(dat)

### Pulling data and TTLs for each mouse

In [ ]:
# Stream / TTL names come from each session's SESSIONS[...]["photometry"] entry.

alldat = []

for f in range(len(masterdat)):
    block = masterdat[f]["data"]
    spec = masterdat[f]["photometry"]

    if masterdat[f]["mouse1"] != "0000":
        s1 = spec["mouse1"]
        g1 = _tdt_stream(block, s1["green_stream"])
        dat1 = {}
        dat1["mouseID"] = masterdat[f]["mouse1"]
        dat1["green"] = g1.data
        dat1["TTL_timestamps"] = _tdt_epoc_onset(block, s1["ttl_epoc"])
        dat1["sampling_rate"] = g1.fs
        dat1["acq_day"] = masterdat[f]["acq_day"]
        dat1["restriction_state"] = masterdat[f]["restriction_state"]
        dat1["session_label"] = masterdat[f].get("session_label", str(dat1["acq_day"]))
        alldat.append(dat1)

    if masterdat[f]["mouse2"] != "0000":
        s2 = spec["mouse2"]
        g2 = _tdt_stream(block, s2["green_stream"])
        dat2 = {}
        dat2["mouseID"] = masterdat[f]["mouse2"]
        dat2["green"] = g2.data
        dat2["TTL_timestamps"] = _tdt_epoc_onset(block, s2["ttl_epoc"])
        dat2["sampling_rate"] = g2.fs
        dat2["acq_day"] = masterdat[f]["acq_day"]
        dat2["restriction_state"] = masterdat[f]["restriction_state"]
        dat2["session_label"] = masterdat[f].get("session_label", str(dat2["acq_day"]))
        alldat.append(dat2)

# Get time series in seconds

for f in range(len(alldat)):
    alldat[f]['time'] = (np.arange(1,len(alldat[f]['green'])+1))/alldat[f]['sampling_rate']

### Remove artifact at start and downsample

In [ ]:
# Trim start of signal
    
t = 8 # time threshold below which we will discard

for f in range(len(alldat)):
    indA = np.argmax(alldat[f]['time'] > t) # find first index of when time crosses threshold
    alldat[f]['time'] = alldat[f]['time'][indA:] # reformat vector to only include allowed time
    alldat[f]['trimmed_green'] = alldat[f]['green'][indA:]

In [ ]:
# Downsample and average 

N = 100 # Average every N samples into 1 value

for f in range(len(alldat)):
    F_green=[]
    for i in range(0, len(alldat[f]['trimmed_green']), N):
        small_list = np.mean(alldat[f]['trimmed_green'][i:i+N])
        F_green.append(small_list)
    alldat[f]['downsampled_green'] = np.array(F_green)
    
    alldat[f]['downsampled_time'] = alldat[f]['time'][::N]
       

### Fit double exponential to each curve and use to correct for bleaching

In [ ]:
# The double exponential curve we are going to fit.
def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    '''Compute a double exponential function with constant offset.
    Parameters:
    t       : Time vector in seconds.
    const   : Amplitude of the constant offset. 
    amp_fast: Amplitude of the fast component.  
    amp_slow: Amplitude of the slow component.  
    tau_slow: Time constant of slow component in seconds.
    tau_multiplier: Time constant of fast component relative to slow. 
    '''
    tau_fast = tau_slow*tau_multiplier
    return const+amp_slow*np.exp(-t/tau_slow)+amp_fast*np.exp(-t/tau_fast)

def get_bounds(trace, timecourse, tau_slow_min=0.0001, tau_slow_max=30):

    signal = trace
    
    # Amplitude bounds
    amp_min = 0  # Assuming amplitude cannot be negative
    amp_max = 2 * np.max(signal)  # Allowing for some flexibility

    # Time constant bounds based on the duration of the experiment
    time_constant_min = tau_slow_min * (timecourse[-1] - timecourse[0])  # Minimum time constant
    time_constant_max = tau_slow_max * (timecourse[-1] - timecourse[0])  # Maximum time constant

    # Offset bounds
    offset_min = np.min(signal) if np.min(signal) < 0 else 0  # Adjust based on signal characteristics
    offset_max = np.max(signal)

    return (
        [amp_min, amp_min, amp_min, time_constant_min, offset_min],
        [amp_max, amp_max, amp_max, time_constant_max, offset_max]
    )

In [ ]:
for f in range(len(alldat)):

# Assign the downsampled curves to variable names. 
    
    green=alldat[f]['downsampled_green']
    time=alldat[f]['downsampled_time']

    # Fit curve to green signal.
    max_sig = np.max(green)
    inital_params = [max_sig/2, max_sig/4, max_sig/4, 3600, 0.1]
    bounds = get_bounds(green, time)
    green_parms, parm_cov = curve_fit(double_exponential, time, green, 
                                    p0=inital_params, bounds=bounds, maxfev=10000)
    green_expfit = double_exponential(time, *green_parms)
    alldat[f]['expfitgreen'] = green_expfit

    #plot fits over denoised data
    fig,ax1=plt.subplots()  
    plot1=ax1.plot(time, green, 'g', label='green')
    plot3=ax1.plot(time, green_expfit, color='k', linewidth=1.5, label='Exponential fit') 


    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Green Signal (V)', color='g')
    ax1.set_title(alldat[f]['mouseID'])

    lines = plot1 + plot3
    labels = [l.get_label() for l in lines]  
    legend = ax1.legend(lines, labels, loc='upper right'); 
   


In [ ]:
#Subtract curve to correct for bleaching

for f in range(len(alldat)):
    green_detrended = alldat[f]['downsampled_green'] - alldat[f]['expfitgreen']
    alldat[f]['detrended_green'] = green_detrended

    time=alldat[f]['downsampled_time']

    fig,ax1=plt.subplots()  
    plot1=ax1.plot(time, green_detrended, 'g', label='green')

    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Green Signal (V)', color='g')
    ax1.set_title(alldat[f]['mouseID'])

    lines = plot1
    labels = [l.get_label() for l in lines]  
    legend = ax1.legend(lines, labels, loc='upper right'); 
     
 

In [ ]:
#renaming to fit with downstream code

for f in range(len(alldat)):
    alldat[f]['corrected_green']=alldat[f]['detrended_green']

### Z scoring

In [ ]:
#pulling out just mice with photometry signals
photom_trial_list = all_trial_list[all_trial_list['group'] == 'GCaMP']
photom_trial_list = photom_trial_list.reset_index(drop=True)

In [ ]:
# Time window settings
PRE_TIME = 10  # seconds before event onset
POST_TIME = 20  # seconds after
fs_plot = alldat[0]["sampling_rate"] / N
TRANGE_plot = [-1 * PRE_TIME * np.floor(fs_plot), POST_TIME * np.floor(fs_plot)]
npts_plot = int(TRANGE_plot[1] - TRANGE_plot[0])
peri_time = np.arange(npts_plot) / fs_plot - PRE_TIME * np.floor(fs_plot) / fs_plot

#### getting green sensor snips

In [ ]:
# Getting green sensor snips for each trial 
for f in range(len(alldat)):
    Beh_t = alldat[f]["TTL_timestamps"]
    trace = alldat[f]["corrected_green"]
    time_vec = alldat[f]["downsampled_time"]
    trace_len = len(trace)
    mouse_id = alldat[f]["mouseID"]
    day = alldat[f]["acq_day"]
    restriction_state = alldat[f]["restriction_state"]

    fs = alldat[f]["sampling_rate"] / N
    TRANGE = [-1 * PRE_TIME * np.floor(fs), POST_TIME * np.floor(fs)]
    correct_size = int(TRANGE[1] - TRANGE[0])

    trials = len(Beh_t)
    dFF_snips = []

    for i in range(trials):
        after_ttl = time_vec > Beh_t[i]

        # Catches for TTL outside recording or insufficient pre/post window
        if not np.any(after_ttl):
            dFF_snips.append([])
            print(f"{mouse_id} day {day} rs {restriction_state} trial {i}: skipped (after_recording)")
            continue

        onset_ind = int(np.argmax(after_ttl))
        pre_ind = onset_ind + int(TRANGE[0])
        post_ind = onset_ind + int(TRANGE[1])

        if pre_ind < 0:
            dFF_snips.append([])
            print(f"{mouse_id} day {day} rs {restriction_state} trial {i}: skipped (pre_window)")
            continue

        if post_ind > trace_len:
            dFF_snips.append([])
            print(f"{mouse_id} day {day} rs {restriction_state} trial {i}: skipped (post_window)")
            continue

        snip = trace[pre_ind:post_ind]

        if len(snip) != correct_size:
            dFF_snips.append([])
            print(
                f"{mouse_id} day {day} rs {restriction_state} trial {i}: "
                f"skipped (bad_length, got {len(snip)}, expected {correct_size})"
            )
        else:
            dFF_snips.append(snip)

    alldat[f]["Stim_snips"] = dFF_snips

    # Confirm snip count matches TTL count and lick trial count for this session
    n_snips = len(alldat[f]["Stim_snips"])
    n_ttls = len(alldat[f]["TTL_timestamps"])
    n_lick_trials = photom_trial_list.loc[
        (photom_trial_list["mouse_id"] == mouse_id)
        & (photom_trial_list["day"] == day)
        & (photom_trial_list["restriction_state"] == restriction_state)
    ].shape[0]

    print(
        f"{mouse_id} | day {day} | rs {restriction_state} | "
        f"n_snips={n_snips} n_ttls={n_ttls} n_lick_trials={n_lick_trials}"
    )

In [ ]:
# putting snips into trial dataframe (match mouse_id + day + restriction_state)

rows = [None] * len(photom_trial_list)

for f in range(len(alldat)):
    stim_snips = alldat[f]["Stim_snips"]
    mouse_id = alldat[f]["mouseID"]
    day = alldat[f]["acq_day"]
    restriction_state = alldat[f]["restriction_state"]

    idxs = photom_trial_list.index[
        (photom_trial_list["mouse_id"] == mouse_id)
        & (photom_trial_list["day"] == day)
        & (photom_trial_list["restriction_state"] == restriction_state)
    ].tolist()
    idxs_sorted = sorted(idxs)

    for i, snip in enumerate(stim_snips):
        if i < len(idxs_sorted):
            rows[idxs_sorted[i]] = snip

photom_trial_list["green_snips"] = rows

In [ ]:
#custom function to ignore empty arrays when averaging
def mean_ignore_empty(arrays):
    non_empty_arrays = [arr for arr in arrays if len(arr) > 0]
    if not non_empty_arrays:
        return np.nan
    return np.mean(non_empty_arrays, axis=0)

In [ ]:
#Plotting green snips for each mouse

# Group by mouse_id and spout_position, calculate the mean of snips
grouped_df = photom_trial_list.groupby(['mouse_id', 'spout_position','group'])['green_snips'].apply(lambda x: mean_ignore_empty(x.tolist())).reset_index()

# Plotting
# Determine grid size
n_mice = grouped_df['mouse_id'].nunique()
n_cols = 3  # Set number of columns for the grid
n_rows = math.ceil(n_mice / n_cols)  # Calculate required rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))  # Adjust figure size
axes = axes.flatten()  # Flatten in case of multiple rows/columns

# Dictionary to store legend handles for all spout positions
legend_handles = {}

# Iterate over mice and plot
for ax, (mouse_id, data) in zip(axes, grouped_df.groupby('mouse_id')):
    for idx, row in data.iterrows():
        line, = ax.plot(peri_time, row['green_snips'], label=f"Spout {row['spout_position']}")

        # Collect legend handles for all spout positions
        if row['spout_position'] not in legend_handles:
            legend_handles[row['spout_position']] = line

    ax.set_title(f"Green snips for {mouse_id}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean snips")
    ax.set_xlim(-5, 10)

# Hide any unused subplots, but leave one empty for the legend
if n_mice < len(axes):
    legend_ax = axes[n_mice]  # Choose an empty subplot for the legend
    legend_ax.axis("off")  # Hide axis
    
    # Create the legend with all spout positions
    legend_ax.legend(legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False)

    # Hide any remaining unused subplots
    for ax in axes[n_mice+1:]:
        ax.axis('off')

plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()


Calculating z based on combined baseline from all trials

In [ ]:
# Baseline-derived z on green snips: extract baseline across all trials/sessions/restrictions per mouse;
# use that mean and st dev to z-score each full trial snip. Stored as base_snips_zscore in photom_trial_list and alldat.

BASELINE_Z_START, BASELINE_Z_END = -5.0, 0.0
_base_mask_dff = (peri_time >= BASELINE_Z_START) & (peri_time < BASELINE_Z_END)


def _pooled_baseline_mu_sd(mouse_id, df, base_mask):
    """Concatenate baseline samples from every trial for this mouse; return (mean, std) of pooled baseline."""
    chunks = []
    for _, row in df[df["mouse_id"] == mouse_id].iterrows():
        snip = row["green_snips"]
        if snip is None:
            continue
        arr = np.asarray(snip, dtype=float).ravel()
        n = min(arr.size, base_mask.size)
        if n == 0 or not np.any(base_mask[:n]):
            continue
        chunks.append(arr[:n][base_mask[:n]])
    if not chunks:
        return 0.0, 1.0
    pooled = np.concatenate(chunks)
    mu = float(np.mean(pooled))
    sd = float(np.std(pooled))
    if sd == 0 or not np.isfinite(sd):
        sd = 1.0
    return mu, sd

_mu_sd_by_mouse = {
    mid: _pooled_baseline_mu_sd(mid, photom_trial_list, _base_mask_dff)
    for mid in photom_trial_list["mouse_id"].dropna().unique()
}

def _zscore_full_snip(snip, mu, sd):
    if snip is None:
        return None
    arr = np.asarray(snip, dtype=float)
    if arr.size == 0:
        return arr
    return (arr - mu) / sd

photom_trial_list["base_snips_zscore"] = photom_trial_list.apply(
    lambda r: _zscore_full_snip(
        r["green_snips"], *_mu_sd_by_mouse.get(r["mouse_id"], (0.0, 1.0))
    ),
    axis=1,
)

for f in range(len(alldat)):
    mid = alldat[f]["mouseID"]
    mu, sd = _mu_sd_by_mouse.get(mid, (0.0, 1.0))
    stim = alldat[f]["Stim_snips"]
    alldat[f]["base_snips_zscore"] = [_zscore_full_snip(snip, mu, sd) for snip in stim]


In [ ]:
# Plot baseline-derived z scores: one panel per (mouse, restriction state)

RESTRICTION_LABEL = {1: "water restricted", 2: "food restricted"}

grouped_base_z = (
    photom_trial_list.groupby(
        ["mouse_id", "restriction_state", "spout_position", "group"]
    )["base_snips_zscore"]
    .apply(lambda x: mean_ignore_empty(x.tolist()))
    .reset_index(name="mean_base_snip_z")
)

n_panels = grouped_base_z.groupby(["mouse_id", "restriction_state"]).ngroups
n_cols = 3
n_rows = math.ceil(n_panels / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = np.atleast_1d(axes).ravel()

legend_handles = {}

panel_groups = grouped_base_z.groupby(["mouse_id", "restriction_state"], sort=True)
for ax, ((mouse_id, rest), data) in zip(axes, panel_groups):
    try:
        rlab = RESTRICTION_LABEL.get(int(rest), str(rest))
    except (TypeError, ValueError):
        rlab = str(rest)
    for _, row in data.iterrows():
        line, = ax.plot(
            peri_time,
            row["mean_base_snip_z"],
            label=f"Spout {row['spout_position']}",
        )
        if row["spout_position"] not in legend_handles:
            legend_handles[row["spout_position"]] = line
    ax.set_title(f"{mouse_id} | {rlab} | baseline-z")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean base_snips_zscore")
    ax.set_xlim(-5, 10)

if n_panels < len(axes):
    legend_ax = axes[n_panels]
    legend_ax.axis("off")
    legend_ax.legend(
        legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False
    )
    for ax in axes[n_panels + 1 :]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Plotting cohort mean baseline-z snips: equal weight per mouse, split by group × restriction state

RESTRICTION_LABEL = {1: "water restricted", 2: "food restricted"}

per_mouse_base_z = (
    photom_trial_list.groupby(
        ["group", "restriction_state", "spout_position", "mouse_id"], sort=False
    )["base_snips_zscore"]
    .apply(lambda x: mean_ignore_empty(x.tolist()))
    .reset_index(name="snip")
)

cohort_base_z_df = (
    per_mouse_base_z.groupby(
        ["group", "restriction_state", "spout_position"], sort=True
    )["snip"]
    .apply(lambda x: mean_ignore_empty(x.tolist()))
    .reset_index(name="mean_snip")
)

n_panels = cohort_base_z_df.groupby(["group", "restriction_state"]).ngroups
n_cols = 3
n_rows = math.ceil(n_panels / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = np.atleast_1d(axes).ravel()

legend_handles = {}

panel_groups = cohort_base_z_df.groupby(["group", "restriction_state"], sort=True)
for ax, ((grp, rest), data) in zip(axes, panel_groups):
    try:
        rlab = RESTRICTION_LABEL.get(int(rest), str(rest))
    except (TypeError, ValueError):
        rlab = str(rest)
    for _, row in data.iterrows():
        line, = ax.plot(
            peri_time,
            row["mean_snip"],
            label=f"Spout {row['spout_position']}",
        )
        if row["spout_position"] not in legend_handles:
            legend_handles[row["spout_position"]] = line
    ax.set_title(f"{grp} | {rlab}\n(mean across mice, baseline-z)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Mean base_snips_zscore")
    ax.set_xlim(-5, 10)

if n_panels < len(axes):
    legend_ax = axes[n_panels]
    legend_ax.axis("off")
    legend_ax.legend(
        legend_handles.values(), legend_handles.keys(), loc="center", fontsize=10, frameon=False
    )
    for ax in axes[n_panels + 1 :]:
        ax.axis("off")

plt.tight_layout()
plt.show()

Export average traces for each mouse

In [ ]:
# Export mean traces per mouse × restriction_state × spout for each snip type (separate Excel files).

TRACE_EXPORTS = [
    ("green_snips", "green_mean.xlsx"),
    ("base_snips_zscore", "base_snips_zscore_mean.xlsx"),
]

os.makedirs(mean_trace_output, exist_ok=True)

for z_col, fname in TRACE_EXPORTS:
    grouped_df = (
        photom_trial_list.groupby(
            ["mouse_id", "restriction_state", "spout_position"], dropna=False
        )[z_col]
        .apply(lambda x: mean_ignore_empty(x.tolist()))
        .reset_index()
    )

    expanded_df = pd.concat(
        [
            grouped_df[["mouse_id", "restriction_state", "spout_position"]],
            grouped_df[z_col].apply(pd.Series),
        ],
        axis=1,
    )

    excel_file_path = os.path.join(mean_trace_output, f"{output_name}_{fname}")
    with pd.ExcelWriter(excel_file_path, engine="xlsxwriter") as writer:
        expanded_df.to_excel(writer, index=False)

Making dataframe for linear model

In [ ]:
# 1) Define time windows (in seconds)
t_base_start, t_base_end = -5.0, 0.0        
t_lick_start, t_lick_end = 0.0, 3.0         
t_post_start, t_post_end = 3.0, 10.0        

# 2) Build masks based on peri_time
base_mask = (peri_time >= t_base_start) & (peri_time < t_base_end)
lick_mask = (peri_time >= t_lick_start) & (peri_time < t_lick_end)
post_mask = (peri_time >= t_post_start) & (peri_time < t_post_end)
lick_and_post_mask = (peri_time >= t_lick_start) & (peri_time < t_post_end)

# Helper that safely averages with a mask
def masked_mean(arr, mask):
    arr = np.asarray(arr)
    if arr.size == 0:
        return np.nan
    # In case arr length differs slightly from peri_time, align by min length
    n = min(arr.size, mask.size)
    if n == 0 or not mask[:n].any():
        return np.nan
    return arr[:n][mask[:n]].mean()

# 3) Create the new columns for green snips (not really dFF, just detrended)
photom_trial_list["dFF_base"] = photom_trial_list["green_snips"].apply(
    lambda a: masked_mean(a, base_mask)
)
photom_trial_list["dFF_licking"] = photom_trial_list["green_snips"].apply(
    lambda a: masked_mean(a, lick_mask)
)
photom_trial_list["dFF_post_licking"] = photom_trial_list["green_snips"].apply(
    lambda a: masked_mean(a, post_mask)
)
photom_trial_list["dFF_lick_and_post_licking"] = photom_trial_list["green_snips"].apply(
    lambda a: masked_mean(a, lick_and_post_mask)
)

# 4) Create the new columns for combined baseline z-scored snips
photom_trial_list["basesnip_z_base"] = photom_trial_list["base_snips_zscore"].apply(
    lambda a: masked_mean(a, base_mask)
)
photom_trial_list["basesnip_z_licking"] = photom_trial_list["base_snips_zscore"].apply(
    lambda a: masked_mean(a, lick_mask)
)
photom_trial_list["basesnip_z_post_licking"] = photom_trial_list["base_snips_zscore"].apply(
    lambda a: masked_mean(a, post_mask)
)
photom_trial_list["basesnip_z_lick_and_post_licking"] = photom_trial_list["base_snips_zscore"].apply(
    lambda a: masked_mean(a, lick_and_post_mask)
)

In [ ]:
# Trial number within each mouse, restriction state, and day (session)
# Sort by spout-extension timestamp to ensure trial_number reflects chronological order
photom_trial_list = photom_trial_list.sort_values(
    ["mouse_id", "restriction_state", "day", "timestamp"],
    kind="mergesort",
).reset_index(drop=True)

photom_trial_list["trial_number"] = (
    photom_trial_list.groupby(
        ["mouse_id", "restriction_state", "day"],
        sort=False,
    ).cumcount()
    + 1
)

In [ ]:
#Converting spout value to a number
code_map = {
    "Null": 1,
    "Water": 2,
    "10% Sucrose": 3,
    "20% Sucrose": 4,
    "Saccharine": 5,
}

photom_trial_list["spout_code"] = photom_trial_list["spout_position"].map(code_map)

# Identifying previous trial within each mouse, restriction state, and day
photom_trial_list["previous_trial"] = (
    photom_trial_list.groupby(
        ["mouse_id", "restriction_state", "day"], sort=False
    )["spout_code"]
    .shift(1)
    .fillna(0)
    .astype(int)
)

In [ ]:
#pulling out needed columns
cols = [
    "mouse_id",
    "day",
    "trial_number",
    "restriction_state",
    "spout_code",
    "previous_trial",
    "total_licks",
    "dFF_base",
    "dFF_licking",
    "dFF_post_licking",
    "dFF_lick_and_post_licking",
    "basesnip_z_base",
    "basesnip_z_licking",
    "basesnip_z_post_licking",
    "basesnip_z_lick_and_post_licking",
]

linear_model_df = photom_trial_list[cols].copy()

# Reformatting for linear model
wide = linear_model_df.reset_index(drop=False).rename(columns={"index": "trial_row_id"})

id_cols = [
    "trial_row_id",
    "mouse_id",
    "day",
    "trial_number",
    "restriction_state",
    "spout_code",
    "previous_trial",
    "total_licks",
]

long_df = pd.wide_to_long(
    wide,
    stubnames=["dFF", "basesnip_z"],
    i=id_cols,
    j="period",
    sep="_",
    suffix=r".+",
).reset_index()

long_df = long_df.rename(
    columns={
        "dFF": "dFF avg",
        "basesnip_z": "basesnip_z avg",
    }
)

period_order = ["base", "licking", "post_licking", "lick_and_post_licking"]
long_df["period"] = pd.Categorical(long_df["period"], categories=period_order, ordered=True)
long_df = long_df.sort_values(["trial_row_id", "period"], kind="mergesort").reset_index(drop=True)

long_df["licking"] = (long_df["period"] == "licking").astype(int)
long_df["post_licking"] = (long_df["period"] == "post_licking").astype(int)
long_df["lick_and_post_licking"] = (long_df["period"] == "lick_and_post_licking").astype(int)

cols = long_df.columns.tolist()
for c in ("licking", "post_licking", "lick_and_post_licking"):
    cols.remove(c)
period_i = cols.index("period")
for j, c in enumerate(["licking", "post_licking", "lick_and_post_licking"]):
    cols.insert(period_i + 1 + j, c)
long_df = long_df[cols]

base_mask = long_df["period"] == "base"
long_df.loc[base_mask, ["spout_code", "total_licks"]] = 0


In [ ]:
#Exporting linear model dataframe
from pathlib import Path

base = output_name if str(output_name).lower().endswith(".csv") else f"{output_name}.csv"
out_path = Path(data_output) / base

out_path.parent.mkdir(parents=True, exist_ok=True)
long_df.to_csv(out_path, index=False)

In [ ]:
#Stop here. Cells below will combine data from multiple cohorts into one csv for linear model.
raise RuntimeError("Stop here — don’t run cells below yet.")

Loading in csvs to combine

In [ ]:
data_dir = Path(data_output)  

csv_paths = sorted(data_dir.glob("*.csv"))

frames = []
for path in csv_paths:
    df = pd.read_csv(path)
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
print(combined["mouse_id"].unique())

In [ ]:
#export combined csv

out_path = Path("./combined.csv")
combined.to_csv(out_path, index=False)